In [ ]:
import os
import json
import requests
from pathlib import Path
from datetime import datetime
import xml.etree.ElementTree as ET
from xml.dom import minidom

# ─── SAM OUTPUT & ANNOTATION OUTPUT ───────────────────────────────────────────
SAM_OUTPUT_DIR = "./sam_output"
CVAT_OUTPUT    = "./cvat_import/annotations.xml"
DUMMY_LABEL    = "rock"
LABEL_COLOR    = "#FF6600"

# ─── CVAT CONNECTION ──────────────────────────────────────────────────────────
# Check the task URL in your browser: http://localhost:8080/tasks/N  → N is CVAT_TASK_ID
CVAT_URL      = "http://localhost:8080"
CVAT_USERNAME = "admin"           # ← your CVAT username
CVAT_PASSWORD = "yourpassword"    # ← your CVAT password
CVAT_TASK_ID  = 1                 # ← integer task ID from the CVAT URL

# ─── TASK METADATA (for XML meta block only, not used for matching) ───────────
TASK_NAME = "Rock Segmentation"

os.makedirs(Path(CVAT_OUTPUT).parent, exist_ok=True)
print(f"SAM output  : {SAM_OUTPUT_DIR}")
print(f"CVAT output : {CVAT_OUTPUT}")
print(f"CVAT task   : {CVAT_URL}/tasks/{CVAT_TASK_ID}")


In [ ]:
def get_cvat_frame_map(cvat_url: str, task_id: int, username: str, password: str) -> dict:
    """
    Fetch the exact frame list from CVAT for a task via REST API.

    CVAT returns frames as an ordered list — the frame index is the
    position in that list (0-based), not a field inside each item.
    """
    url  = f"{cvat_url}/api/tasks/{task_id}/data/meta"
    resp = requests.get(url, auth=(username, password), timeout=30)

    if resp.status_code == 401:
        raise PermissionError("CVAT credentials incorrect. Check CVAT_USERNAME / CVAT_PASSWORD.")
    if resp.status_code == 404:
        raise FileNotFoundError(f"Task {task_id} not found in CVAT. Check CVAT_TASK_ID.")
    resp.raise_for_status()

    frames = resp.json()["frames"]
    mapping = {}

    for idx, f in enumerate(frames):   # idx = frame position in list (0-based)
        cvat_name = f["name"]          # exact name CVAT stored
        stem      = Path(cvat_name).stem
        basename  = Path(cvat_name).name

        # Primary key: exact CVAT stored name (may include subpath from ZIP)
        mapping[cvat_name] = (idx, cvat_name)
        # Fallback: basename with extension
        if basename not in mapping:
            mapping[basename] = (idx, cvat_name)
        # Fallback: basename without extension
        if stem not in mapping:
            mapping[stem] = (idx, cvat_name)

    print(f"Fetched {len(frames)} frames from CVAT task {task_id}:")
    for idx, f in enumerate(frames[:5]):
        print(f"  [{idx:3d}]  {f['name']}")
    if len(frames) > 5:
        print(f"  ... and {len(frames) - 5} more")

    return mapping


# ── Fetch now — fail early if credentials / task ID are wrong ─────────────────
frame_map = get_cvat_frame_map(CVAT_URL, CVAT_TASK_ID, CVAT_USERNAME, CVAT_PASSWORD)


In [ ]:
def points_to_cvat_string(flat_points: list) -> str:
    """Convert flat list [x1,y1,x2,y2,...] to CVAT points string 'x1.00,y1.00;x2.00,y2.00;...'"""
    pairs = []
    it = iter(flat_points)
    for x in it:
        y = next(it)
        pairs.append(f"{float(x):.2f},{float(y):.2f}")
    return ";".join(pairs)


def build_cvat_xml(
    sam_output_dir : str,
    dummy_label    : str,
    label_color    : str,
    task_name      : str,
    task_id        : int,
    frame_map      : dict,     # from get_cvat_frame_map()
) -> ET.Element:
    """
    Build a CVAT XML 1.1 annotation tree from per-image JSON files.

    Uses `frame_map` to resolve each image's exact CVAT frame index and
    stored name — guaranteeing the import matches CVAT's internal state.

    Matching priority for each JSON record's `filename`:
      1. Exact CVAT-stored name  (e.g. "images/foo.jpg" from ZIP upload)
      2. Basename with extension (e.g. "foo.jpg")
      3. Basename without extension (e.g. "foo")
    """
    json_files = sorted(Path(sam_output_dir).glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"No JSON files found in {sam_output_dir}")

    print(f"Found {len(json_files)} JSON files to convert.")

    # ─ Root element ──────────────────────────────────────────────────────────
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"

    # ─ Meta block ────────────────────────────────────────────────────────────
    meta  = ET.SubElement(root, "meta")
    task  = ET.SubElement(meta, "task")
    ET.SubElement(task, "id").text   = str(task_id)
    ET.SubElement(task, "name").text = task_name
    ET.SubElement(task, "size").text = str(len(json_files))
    ET.SubElement(task, "mode").text = "annotation"

    segments = ET.SubElement(task, "segments")
    seg      = ET.SubElement(segments, "segment")
    ET.SubElement(seg, "id").text    = "1"
    ET.SubElement(seg, "start").text = "0"
    ET.SubElement(seg, "stop").text  = str(len(json_files) - 1)

    labels_el = ET.SubElement(task, "labels")
    label_el  = ET.SubElement(labels_el, "label")
    ET.SubElement(label_el, "name").text  = dummy_label
    ET.SubElement(label_el, "color").text = label_color
    ET.SubElement(label_el, "attributes")

    ET.SubElement(meta, "dumped").text = datetime.now().isoformat()

    # ─ Image blocks ──────────────────────────────────────────────────────────
    total_polygons  = 0
    skipped_nomask  = 0
    skipped_nomatch = 0

    for json_path in json_files:
        with open(json_path) as f:
            record = json.load(f)

        if record["n_masks"] == 0:
            skipped_nomask += 1
            continue

        # Resolve CVAT frame index and exact stored name
        filename = record["filename"]
        stem     = Path(filename).stem

        if filename in frame_map:
            frame_idx, cvat_name = frame_map[filename]
        elif stem in frame_map:
            frame_idx, cvat_name = frame_map[stem]
        else:
            print(f"  [WARN] '{filename}' not found in CVAT task — skipped.")
            print(f"         Check that this image was uploaded to task {task_id}.")
            skipped_nomatch += 1
            continue

        img_el = ET.SubElement(root, "image")
        img_el.set("id",     str(frame_idx))   # CVAT's actual frame index
        img_el.set("name",   cvat_name)         # CVAT's exact stored name
        img_el.set("width",  str(record["width"]))
        img_el.set("height", str(record["height"]))

        for mask in record["masks"]:
            for poly_pts in mask["polygons"]:
                if len(poly_pts) < 6:
                    continue
                poly_el = ET.SubElement(img_el, "polygon")
                poly_el.set("label",    dummy_label)
                poly_el.set("points",   points_to_cvat_string(poly_pts))
                poly_el.set("z_order",  "0")
                poly_el.set("occluded", "0")
                total_polygons += 1

    print(f"\nSummary:")
    print(f"  Images annotated       : {len(json_files) - skipped_nomask - skipped_nomatch}")
    print(f"  Skipped (0 masks)      : {skipped_nomask}")
    print(f"  Skipped (not in CVAT)  : {skipped_nomatch}")
    print(f"  Total polygons written : {total_polygons}")

    return root


def write_pretty_xml(element: ET.Element, output_path: str):
    raw    = ET.tostring(element, encoding="unicode")
    parsed = minidom.parseString(raw)
    pretty = parsed.toprettyxml(indent="  ", encoding="utf-8")
    with open(output_path, "wb") as f:
        f.write(pretty)
    print(f"\nAnnotation file saved: {output_path}")
    print(f"File size: {Path(output_path).stat().st_size / 1024:.1f} KB")


# ─── RUN ──────────────────────────────────────────────────────────────────────
root = build_cvat_xml(
    sam_output_dir = SAM_OUTPUT_DIR,
    dummy_label    = DUMMY_LABEL,
    label_color    = LABEL_COLOR,
    task_name      = TASK_NAME,
    task_id        = CVAT_TASK_ID,
    frame_map      = frame_map,    # fetched from CVAT API in previous cell
)

write_pretty_xml(root, CVAT_OUTPUT)


In [ ]:
def validate_cvat_xml(xml_path: str):
    """Quick structural validation of the generated XML."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    version  = root.findtext("version")
    images   = root.findall("image")
    polygons = root.findall(".//polygon")
    labels   = {el.findtext("name") for el in root.findall(".//label")}

    print(f"CVAT XML Validation")
    print(f"  Version   : {version}")
    print(f"  Images    : {len(images)}")
    print(f"  Polygons  : {len(polygons)}")
    print(f"  Labels    : {labels}")

    # Check for common issues
    issues = []
    for img in images:
        if not img.get("name"):
            issues.append(f"  [WARN] <image> missing 'name' attribute: id={img.get('id')}")
        polys = img.findall("polygon")
        for p in polys:
            pts = p.get("points", "")
            n_coords = len(pts.split(";"))
            if n_coords < 3:
                issues.append(
                    f"  [WARN] Polygon with < 3 points in {img.get('name')}"
                )

    if issues:
        print("\nIssues found:")
        for issue in issues:
            print(issue)
    else:
        print("\n  No issues found. File is ready for CVAT import.")

validate_cvat_xml(CVAT_OUTPUT)